In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import httpx
import json

load_dotenv()

client = OpenAI(
    http_client=httpx.Client(verify=False)
)
messages=[]

In [2]:
def get_current_weather(city: str):
    return f"The weather in {city} is sunny"

FUNCTIONS = {
    "get_current_weather": get_current_weather
}

In [3]:
TOOLS=[
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to get the current weather in"
                    }
                },
                "required": ["city"]
            }
        }
    }
]


In [4]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append({
        "role": "assistant", 
        "content": message.content or "",
        "tool_calls": [{
            "id": tool_call.id,
            "type": "function",
            "function": {   
                "name": tool_call.function.name,
                "arguments": tool_call.function.arguments
            }
        } for tool_call in message.tool_calls]})
        
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name 
            arguments = tool_call.function.arguments

            print(f"Function: {function_name}")
            print(f"Arguments: {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_call = FUNCTIONS.get(function_name)
            result = function_to_call(**arguments)

            messages.append({
                "role": "tool",
                "content": result,
                "name": function_name,
                "tool_call_id": tool_call.id
            })
            call_ai()

        else:
            messages.append({"role": "assistant", "content": message.content})
        

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
        tool_choice="auto"
    )
    process_ai_response(response.choices[0].message)
    print(response)
    message = response.choices[0].message.content
    messages.append({"role": "assistant", "content": message})
    print(f"AI: {message}")


In [5]:
while True:
    message = input("Sned a message to the LLM ")
    if message == "exit" or message == "quit":
        break
    else :
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()
    

User: My name is Jin
ChatCompletion(id='chatcmpl-Dql7bjPXRV6iZM2cs7STrSYTQOl2g', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Nice to meet you, Jin! How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1781466563, model='gpt-4o-mini-2024-07-18', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_a1aad5391f', usage=CompletionUsage(completion_tokens=15, prompt_tokens=64, total_tokens=79, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
AI: Nice to meet you, Jin! How can I assist you today?
User: WHat is my name again?
ChatCompletion(id='chatcmpl-Dql8KX4PkZwo2fRBwwqEKUv05qjuU', choices=[Choice(finish_reason='stop', index=0, logprobs=None, m

In [6]:
messages

[{'role': 'user', 'content': 'My name is Jin'},
 {'role': 'assistant',
  'content': 'Nice to meet you, Jin! How can I assist you today?'},
 {'role': 'user', 'content': 'WHat is my name again?'},
 {'role': 'assistant', 'content': 'Your name is Jin.'},
 {'role': 'user', 'content': 'what is the weather in berlin?'},
 {'role': 'assistant',
  'content': '',
  'tool_calls': [{'id': 'call_u1qHfgudYzyMmsdRy1nbXwbm',
    'type': 'function',
    'function': {'name': 'get_current_weather',
     'arguments': '{"city":"Berlin"}'}}]},
 {'role': 'tool',
  'content': 'The weather in Berlin is sunny',
  'name': 'get_current_weather',
  'tool_call_id': 'call_u1qHfgudYzyMmsdRy1nbXwbm'},
 {'role': 'assistant', 'content': 'The weather in Berlin is sunny.'},
 {'role': 'assistant', 'content': None},
 {'role': 'assistant', 'content': None}]